# Demand Forecasting and Inventory Optimization for Small Businesses
## Phase 1: Data Cleaning and Exploratory Data Analysis (EDA)
**Author:** Member 1 (Mugisha Elisa)  
**Dataset:** UCI Online Retail Dataset  
**Fixes (3 Nov 2026, Molemi Molokoane):** removed orders that were later cancelled, grouped the top 10 by `StockCode`, fixed the weekly table (it was all zeros), and pointed the file paths at the shared team Drive folder.  

---

### Project Context & Objectives
This notebook represents the foundational work for our team's Data Science project. As Member 1, my responsibility is to transform the raw retail transaction dataset into clean, reliable, and well-documented outputs.

Our team members will build on these outputs:
- **Member 2:** SQL and Sales Analysis
- **Member 3:** Demand Forecasting & Machine Learning Model Evaluation
- **Member 4:** Power BI Dashboard and Visualizations

Let's get started by importing our tools!

## 1. Import Libraries
We import standard libraries essential for data manipulation, numerical calculation, and visualization.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style for professional visualizations
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries successfully imported!")

## 2. Load the Dataset
We mount Google Drive to access the dataset directly from our project directory and load it safely using `pandas`.

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
    print("Google Drive successfully mounted!")
except Exception as e:
    print(f"Drive mounting skipped or encountered an error: {e}")

# User Input for dataset path
default_path = "/content/drive/MyDrive/WTM-Data-Science-Group-B/data/raw/Online Retail.xlsx"
file_path = input(f"Please enter the path to your Excel file (Press Enter to use default: '{default_path}'): ") or default_path

try:
    print("Loading dataset... This may take a moment as the file contains over 500k rows.")
    df_raw = pd.read_excel(file_path)
    print("Dataset successfully loaded!")

    # Display basic dimensions and data types
    print(f"\nDataset Dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
    print("\n--- First 5 Rows ---")
    display(df_raw.head())

    print("\n--- Data Types ---")
    display(df_raw.dtypes)
except Exception as e:
    print(f"\n[Error] Could not load the Excel file. Please check your path: '{file_path}'")
    print(f"Error details: {e}")

## 3. Data Understanding
Before doing any cleaning, let's analyze the properties and characteristics of our raw data.

In [ ]:
# 1. Missing Values
print("--- Missing Values Count ---")
print(df_raw.isnull().sum())

# 2. Duplicate Records
print(f"\nDuplicate Records: {df_raw.duplicated().sum()} rows")

# 3. Negative & Zero Quantities
neg_quantities = (df_raw['Quantity'] <= 0).sum()
print(f"Transactions with Quantity <= 0: {neg_quantities}")

# 4. Cancelled Invoices
cancelled_invoices = df_raw['InvoiceNo'].astype(str).str.startswith('C').sum()
print(f"Cancelled Transactions (InvoiceNo starting with 'C'): {cancelled_invoices}")

# 5. Descriptive Statistics
print("\n--- Descriptive Statistics ---")
display(df_raw.describe(include='all'))

### Column Glossary & Descriptions
- **InvoiceNo**: A 6-digit integral number uniquely assigned to each transaction. If this code starts with letter 'c', it indicates a cancellation.
- **StockCode**: Product item code. A 5-digit integral number uniquely assigned to each distinct product.
- **Description**: Product (item) name.
- **Quantity**: The quantities of each product (item) per transaction. Negative numbers indicate returns/cancellations.
- **InvoiceDate**: Invice Date and time. The day and time when each transaction was generated.
- **UnitPrice**: Product price per unit in sterling.
- **CustomerID**: Customer number. A 5-digit integral number uniquely assigned to each customer.
- **Country**: Country name. The name of the country where each customer resides.

## 4. Data Cleaning
We will perform the following steps while protecting the raw data:
1. Drop duplicate records.
2. Standardize `InvoiceDate` to a proper Datetime format.
3. Convert numeric types.
4. Separate canceled invoices into a dedicated returns dataset, keeping the main dataset exclusively for positive demand.
5. Remove orders that were cancelled later (matched by customer, product and quantity). Without this, two huge orders that were cancelled within minutes (80,995 × PAPER CRAFT, LITTLE BIRDIE and 74,215 × MEDIUM CERAMIC TOP STORAGE JAR) would look like best-sellers.
6. Filter out invalid/non-positive quantities and prices.
7. Add a calculated `Revenue` column.
8. Handle missing CustomerIDs by flagging them, preserving rows for broad item-level demand analysis.

In [ ]:
# Make a copy to protect the raw dataset
df_clean = df_raw.copy()

# 1. Remove duplicate records
df_clean = df_clean.drop_duplicates()

# 2. Convert InvoiceDate to datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# 3. Handle Missing values: Fill missing descriptions with 'Unknown' and missing CustomerIDs with -1 (representing guest checkouts)
df_clean['Description'] = df_clean['Description'].fillna('Unknown')
df_clean['CustomerID'] = df_clean['CustomerID'].fillna(-1).astype(int)

# 4. Separate cancelled invoices (Starting with C)
df_cancelled = df_clean[df_clean['InvoiceNo'].astype(str).str.startswith('C')]
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# 5. Remove orders that were later cancelled (same customer, product and quantity)
canc = df_cancelled[df_cancelled['CustomerID'] != -1][['CustomerID', 'StockCode', 'Quantity']].copy()
canc['Quantity'] = -canc['Quantity']
canc = canc.drop_duplicates()
rows_before = len(df_clean)
df_clean = df_clean.merge(canc, on=['CustomerID', 'StockCode', 'Quantity'], how='left', indicator=True)
df_clean = df_clean[df_clean['_merge'] == 'left_only'].drop(columns='_merge')
print(f"Orders removed because they were cancelled later: {rows_before - len(df_clean)}")

# 6. Keep only valid positive prices and quantities for the main demand forecasting dataset
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

# 7. Calculate Revenue
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['UnitPrice']

print("Data cleaning steps executed successfully!")
print(f"Cleaned dataset size: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns.")

## 5. Exploratory Data Analysis (EDA)
Let's construct the visualizations needed to understand sales performance and demand dynamics.

In [ ]:
# A. Top Selling Products by Quantity
top_qty = df_clean.groupby('Description')['Quantity'].sum().nlargest(10).reset_index()

sns.barplot(data=top_qty, x='Quantity', y='Description', palette='Blues_r')
plt.title('Top 10 Products Sold by Quantity')
plt.xlabel('Total Quantity Sold')
plt.ylabel('Product Description')
plt.tight_layout()
plt.show()

In [ ]:
# B. Top Generating Revenue Products
top_rev = df_clean.groupby('Description')['Revenue'].sum().nlargest(10).reset_index()

sns.barplot(data=top_rev, x='Revenue', y='Description', palette='Greens_r')
plt.title('Top 10 Products Generating Highest Revenue')
plt.xlabel('Total Revenue Generated (£)')
plt.ylabel('Product Description')
plt.tight_layout()
plt.show()

In [ ]:
# C. Monthly Sales Trend Over Time
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M')
trend_df = df_clean.groupby('YearMonth')['Revenue'].sum().reset_index()
trend_df['YearMonth'] = trend_df['YearMonth'].astype(str)

sns.lineplot(data=trend_df, x='YearMonth', y='Revenue', marker='o', linewidth=2.5, color='orange')
plt.title('Monthly Sales Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue (£)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# D. Country Sales Distribution (Excluding UK for visual scaling if UK dominates)
top_countries = df_clean.groupby('Country')['Revenue'].sum().nlargest(10).reset_index()

sns.barplot(data=top_countries, x='Revenue', y='Country', palette='Purples_r')
plt.title('Top 10 Countries by Total Revenue')
plt.xlabel('Total Revenue (£)')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

## 6. Top 10 Products
We extract the core group of the top 10 products by overall sales quantity.

In [ ]:
top_10_summary = df_clean.groupby('StockCode').agg(
    Description=('Description', 'first'),
    TotalQuantity=('Quantity', 'sum'),
    TotalRevenue=('Revenue', 'sum')
).reset_index().nlargest(10, 'TotalQuantity')

display(top_10_summary)

## 7. Weekly Sales Table (Resampling)
For Member 3 (Forecasting), we construct a regular weekly demand series for the top 10 products, ensuring zero-demand weeks are explicitly represented.

In [ ]:
# List of Top 10 StockCodes
top_10_codes = top_10_summary['StockCode'].tolist()

# Filter main dataset
df_top10 = df_clean[df_clean['StockCode'].isin(top_10_codes)].copy()

# Extract Week start dates
df_top10['Week'] = df_top10['InvoiceDate'].dt.to_period('W').dt.start_time

# Aggregate quantities
weekly_agg = df_top10.groupby(['StockCode', 'Week'])['Quantity'].sum().reset_index()

# Create a complete combination matrix of all weeks and all top 10 products
all_weeks = pd.date_range(start=df_top10['Week'].min(), end=df_top10['Week'].max(), freq='W-MON')
idx = pd.MultiIndex.from_product(
    [top_10_codes, all_weeks],
    names=['StockCode', 'Week']
)

# Map stockcode to description
code_to_desc = dict(zip(top_10_summary['StockCode'], top_10_summary['Description']))

# Reindex to insert zeros for missing weeks
weekly_sales = pd.DataFrame(index=idx).reset_index()
weekly_sales['Description'] = weekly_sales['StockCode'].map(code_to_desc)

# Merge with aggregate results
weekly_sales = weekly_sales.merge(weekly_agg, on=['StockCode', 'Week'], how='left')
weekly_sales['WeeklyQuantity'] = weekly_sales['Quantity'].fillna(0).astype(int)
weekly_sales = weekly_sales.drop(columns=['Quantity'])

# Sort
weekly_sales = weekly_sales.sort_values(by=['StockCode', 'Week'])
display(weekly_sales.head(15))

## 8. Data Quality Checks
We perform a final audit to ensure clean, valid data output.

In [ ]:
print(f"Missing Values: {weekly_sales.isnull().sum().sum()}")
print(f"Duplicate Rows: {weekly_sales.duplicated().sum()}")
print(f"Minimum Quantity Checked (Should be >= 0): {weekly_sales['WeeklyQuantity'].min()}")
print(f"Weekly Aggregated Total Quantity: {weekly_sales['WeeklyQuantity'].sum()}")
print(f"Raw Top-10 Underlying Quantity: {df_top10['Quantity'].sum()}")

## 9. Export Files
Save outputs directly into Google Drive.

In [ ]:
output_dir = "/content/drive/MyDrive/WTM-Data-Science-Group-B/data/processed"
os.makedirs(output_dir, exist_ok=True)

try:
    df_clean.to_csv(os.path.join(output_dir, 'cleaned_retail_data.csv'), index=False)
    weekly_sales.to_csv(os.path.join(output_dir, 'weekly_sales.csv'), index=False)
    top_10_summary.to_csv(os.path.join(output_dir, 'top_10_products.csv'), index=False)

    # Generate quick summary statistics to CSV
    summary_stats = df_clean.describe(include='all')
    summary_stats.to_csv(os.path.join(output_dir, 'eda_summary.csv'))

    print(f"All processed CSV files successfully exported to: {output_dir}")
except Exception as e:
    print(f"Export failed: {e}")

## 10. Report & Documentation

### Executive Summary
- **Cleaned Dataset Overview**: About 4% of raw records (22,503 of 541,909) were removed: duplicates, cancellations, orders that were later cancelled, and rows with invalid quantities or prices. Crucially, the records with missing `CustomerID` have been retained using placeholders to avoid skewing our sales aggregate demand analyses.
- **Top 10 Products**: Led by "SMALL POPCORN HOLDER", "WORLD WAR 2 GLIDERS ASSTD DESIGNS" and "JUMBO BAG RED RETROSPOT".
- **Cancelled bulk orders**: Two very large orders (80,995 and 74,215 units) were cancelled within minutes. Removing orders with a matching cancellation stops them from distorting the top 10 and the forecasts.
- **Limitations**: The dataset spans only about a single year (Dec 2010 - Dec 2011), meaning full year-over-year seasonality effects will be challenging to capture perfectly.